# LSTM для классификации текстов

## 1. Загрузка необходимых библиотек и данных

### 1.1 Импорты

In [101]:
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from collections import Counter
from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups

import nltk
from nltk.corpus import stopwords

In [102]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1.2 Emotion

In [103]:
dataset_emotion = load_dataset('emotion', split='train+test')

print(f"emotion size: {len(dataset_emotion)}")

emotion size: 18000


### 1.3 News

In [104]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

dataset_news = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

print(f"news train size: {len(dataset_news.data)}")

news train size: 3906


## 2. Параметры

In [105]:
torch.manual_seed(42)
np.random.seed(42)

In [106]:
EMBEDDING_DIM = 100
HIDDEN_DIM = 64
NUM_LAYERS = 3
DROPOUT = 0.2
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
MAX_LEN = {
    'emotion': 100,
    'news': 300
}
TOP_K = 20000

In [107]:
if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cuda


## 3. Предобработка

### 3.1 Удаление шума(для news)

In [108]:
def clean_noise(text):
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum/len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r"[^\w\s.!?]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 3.2 Токенизация

In [109]:
stop_words = set(stopwords.words("english"))

def tokenize_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w not in stop_words]
    return words

In [110]:
preprocessors= {
    'tokenization': tokenize_text
}

### 3.3 Применение предобработки

In [111]:
datasets = {'emotion': dataset_emotion, 'news': dataset_news}
datasets_texts = {}
datasets_labels = {}

In [112]:
datasets_texts['emotion'] = datasets['emotion']['text']
datasets_texts['news'] = [clean_noise(text) for text in datasets['news'].data]

In [113]:
datasets_labels['emotion'] = datasets['emotion']['label']
datasets_labels['news'] = datasets['news'].target

In [114]:
preprocessed_datasets = {'emotion': {}, 'news': {}}

In [115]:
for prep_name, prep_func in preprocessors.items():
    for dataset_name in datasets.keys():
        print(f"Processing {dataset_name} with {prep_name}")
        preprocessed_datasets[dataset_name][prep_name] = [prep_func(text) for text in datasets_texts[dataset_name]]

Processing emotion with tokenization
Processing news with tokenization


## 3.4 Построение словаря

In [116]:
counters = {}
word2idx = {}
X_data = {}
y_data = {}

for dataset_name in datasets.keys():
    counters[dataset_name] = {}
    word2idx[dataset_name] = {}
    X_data[dataset_name] = {}
    y_data[dataset_name] = {}

    for prep_name in preprocessors.keys():
        print(f"Processing {prep_name} with {dataset_name}")

        texts = preprocessed_datasets[dataset_name][prep_name]
        labels = datasets_labels[dataset_name]

        # 1. Строим частотный словарь по всем текстам
        counter = Counter()
        for text in texts:
            counter.update(text)
        counters[dataset_name][prep_name] = counter

        # 2. Создаём word2idx (TOP_K самых частых слов + OOV)
        most_common = counter.most_common(TOP_K-1)
        w2i = {word: i+1 for i, (word, _) in enumerate(most_common)}
        w2i['<OOV>'] = len(w2i)+1
        word2idx[dataset_name][prep_name] = w2i
        vocab_size = len(w2i)+1  # +1 для padding=0
        print(f"vocab size: {vocab_size}")

        # 3. Преобразуем тексты в последовательности индексов
        sequences = []
        for text in texts:
            swq = [w2i.get(w,w2i['<OOV>']) for w in text]
            sequences.append(swq)

        # 4. Паддинг до MAX_LEN[dataset_name]
        max_len = MAX_LEN[dataset_name]
        padded = []

        for seq in sequences:
            if len(seq) >= max_len:
                padded.append(seq[:max_len])
            else:
                padded.append(seq + [0] * (max_len-len(seq)))
        # 5. Преобразуем в тензоры PyTorch
        X_data[dataset_name][prep_name] = torch.tensor(padded)
        y_data[dataset_name][prep_name] = torch.tensor(labels)

Processing tokenization with emotion
vocab size: 16035
Processing tokenization with news
vocab size: 20001


## 4. Классы для LSTM

In [117]:
class TextDataset(Dataset):
    def __init__(self,X,y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self,idx):
        return self.X[idx], self.y[idx]

In [118]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, num_layers=1, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        last_hidden = hidden[-1]
        out = self.dropout(last_hidden)
        out = self.fc(out)
        return out

## 5. Функция обучения

In [119]:
def train(X_train, y_train, X_test, y_test, vocab_size, num_classes,emb_dim=EMBEDDING_DIM, hid_dim=HIDDEN_DIM, n_layers=NUM_LAYERS, dropout=DROPOUT, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    model = LSTMClassifier(vocab_size, emb_dim, hid_dim, num_classes, n_layers, dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_dataset = TextDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for bx,by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            outputs = model(bx)
            loss = criterion(outputs, by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * bx.size(0)
        avg_loss = total_loss / len(X_train)

        model.eval()
        with torch.no_grad():
            pred = model(X_test.to(device)).argmax(1)
            acc = (pred == y_test.to(device)).float().mean().item()
            print (f"Epoch{epoch+1}, loss {avg_loss:.4f}, test acc {acc:.4f}")

    with torch.no_grad():
        pred = model(X_test.to(device)).argmax(1)
    f1 = f1_score(y_test.cpu().numpy(), pred.cpu().numpy(), average='micro')
    print(f"f1-micro: {f1:.4f}")
    return model

## 6. Работа модели

In [120]:
results = []

for dataset_name in datasets.keys():
    for preprocessor_name in preprocessors.keys():
        X = X_data[dataset_name][preprocessor_name]
        y = y_data[dataset_name][preprocessor_name]

        X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)

        vocab_size = len(word2idx[dataset_name][preprocessor_name]) + 1
        num_classes = len(torch.unique(y_train))


        model = train(X_train,y_train,X_test,y_test,
                      vocab_size=vocab_size, num_classes=num_classes)

        model.eval()
        with torch.no_grad():
            pred = model(X_test.to(device)).argmax(1)
            acc = (pred == y_test.to(device)).float().mean().item()
            f1_micro = f1_score(y_test.cpu().numpy(), pred.cpu().numpy(), average='micro')

        results.append({
            'Dataset': dataset_name,
            'Preprocessing': preprocessor_name,
            'Test Accuracy': acc,
            'F1-micro': f1_micro,
            'Vocabulary size': vocab_size,
            'Num classes': num_classes
        })

C:\Users\alesh\AppData\Local\Temp\ipykernel_17816\3907360310.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.X = torch.tensor(X)
C:\Users\alesh\AppData\Local\Temp\ipykernel_17816\3907360310.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y)


Epoch1, loss 1.5911, test acc 0.3364
Epoch2, loss 1.5778, test acc 0.3364
Epoch3, loss 1.5762, test acc 0.3364
Epoch4, loss 1.5766, test acc 0.3364
Epoch5, loss 1.5768, test acc 0.3364
Epoch6, loss 1.5761, test acc 0.3364
Epoch7, loss 1.5750, test acc 0.3364
Epoch8, loss 1.5756, test acc 0.3364
Epoch9, loss 1.5756, test acc 0.3364
Epoch10, loss 1.5763, test acc 0.3364
Epoch11, loss 1.5762, test acc 0.3364
Epoch12, loss 1.5746, test acc 0.3364
Epoch13, loss 1.5756, test acc 0.3364
Epoch14, loss 1.5753, test acc 0.3364
Epoch15, loss 1.5759, test acc 0.3364
Epoch16, loss 1.5752, test acc 0.3364
Epoch17, loss 1.5742, test acc 0.3364
Epoch18, loss 1.5750, test acc 0.3364
Epoch19, loss 1.5746, test acc 0.3364
Epoch20, loss 1.5754, test acc 0.3364
f1-micro: 0.3364


C:\Users\alesh\AppData\Local\Temp\ipykernel_17816\3907360310.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.X = torch.tensor(X)
C:\Users\alesh\AppData\Local\Temp\ipykernel_17816\3907360310.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y)


Epoch1, loss 1.3879, test acc 0.2532
Epoch2, loss 1.3866, test acc 0.2558
Epoch3, loss 1.3814, test acc 0.2532
Epoch4, loss 1.3728, test acc 0.2583
Epoch5, loss 1.3635, test acc 0.2545
Epoch6, loss 1.3584, test acc 0.2506
Epoch7, loss 1.3519, test acc 0.2519
Epoch8, loss 1.3482, test acc 0.2558
Epoch9, loss 1.3435, test acc 0.2583
Epoch10, loss 1.3418, test acc 0.2570
Epoch11, loss 1.3425, test acc 0.2545
Epoch12, loss 1.3409, test acc 0.2570
Epoch13, loss 1.3398, test acc 0.2494
Epoch14, loss 1.3368, test acc 0.2545
Epoch15, loss 1.3370, test acc 0.2519
Epoch16, loss 1.3358, test acc 0.2519
Epoch17, loss 1.3354, test acc 0.2506
Epoch18, loss 1.3344, test acc 0.2583
Epoch19, loss 1.3348, test acc 0.2545
Epoch20, loss 1.3348, test acc 0.2545
f1-micro: 0.2545


## 7. Результаты

In [121]:
df_results = pd.DataFrame(results)
display(df_results)

,Dataset,Preprocessing,Test Accuracy,F1-micro,Vocabulary size,Num classes
0,emotion,tokenization,0.336389,0.336389,16035,6
1,news,tokenization,0.254476,0.254476,20001,4
